# Análise de Fauna

Foca na **fauna**: quanto cada agente joga de cada tipo, o uso da **capacidade habitacional**, a
economia de **O2** (produção sustenta a fauna; excesso de animais é sacrificado), e a contribuição
da fauna ao **score**. Agrupa por **agente** (justo com `randomize_seats`).

Usa `turn_states.csv` (progressão por rodada) e o `results.csv` do mesmo torneio (volume por tipo).

In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()


def find_latest_turn_states(root: Path) -> Path:
    base = root / 'artifacts' / 'tournaments'
    for d in sorted(base.glob('*/'), key=lambda p: p.stat().st_mtime, reverse=True):
        ts = d / 'turn_states.csv'
        if ts.exists() and 'fauna_count' in pd.read_csv(ts, nrows=0).columns:
            return ts
    raise FileNotFoundError('Nenhum turn_states.csv com coluna fauna_count. Rode um torneio novo.')


ts_path = find_latest_turn_states(ROOT)
print('Usando:', ts_path.parent.name)
df = pd.read_csv(ts_path)


def read_meta(csv_path):
    m = csv_path.parent / 'metadata.json'
    return json.loads(m.read_text(encoding='utf-8')).get('tournament_config', {}) if m.exists() else {}

meta = read_meta(ts_path)
if 'agent' not in df.columns:
    df['agent'] = df['player'].map({1: meta.get('agent_p1', 'P1'), 2: meta.get('agent_p2', 'P2')})

# fim-de-rodada por (jogo, jogador, rodada) -> média entre jogos por (rodada, agente)
eor = df.loc[df.groupby(['seed', 'player', 'round'])['turn'].idxmax()]
agg = eor.groupby(['round', 'agent']).mean(numeric_only=True).reset_index()
agents = sorted(agg['agent'].unique())
print('Agentes:', agents, '| jogos:', df['seed'].nunique())


def series(metric, agent):
    s = agg[agg['agent'] == agent].set_index('round')[metric].sort_index()
    return s.index, s.values

## Fauna no board e capacidade habitacional, por agente

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for a in agents:
    x, y = series('fauna_count', a); axes[0].plot(x, y, marker='o', label=a)
axes[0].set_title('Fauna no board (média por rodada)'); axes[0].set_xlabel('rodada'); axes[0].set_ylabel('nº de fauna'); axes[0].legend()
for a in agents:
    x, y = series('habitat_capacity', a); axes[1].plot(x, y, marker='o', linestyle='--', label=f'{a} capacidade')
    x, y = series('fauna_count', a); axes[1].plot(x, y, marker='.', label=f'{a} fauna')
axes[1].set_title('Fauna vs capacidade habitacional'); axes[1].set_xlabel('rodada'); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## Economia de O2 (sustento da fauna)

A produção de O2 é a capacidade de sustento: cada fauna consome 1 O2/rodada e o excesso é sacrificado.
`produced_o2` é o O2 **excedente acumulado** (net, depois do consumo) — cresce quando sobra produção.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for a in agents:
    x, y = series('produced_o2', a); ax.plot(x, y, marker='o', label=f'{a} O2 excedente (acum.)')
ax.set_title('O2 excedente acumulado por rodada'); ax.set_xlabel('rodada'); ax.set_ylabel('O2'); ax.legend()
plt.tight_layout(); plt.show()

## Volume de fauna por tipo e contribuição ao score

Do `results.csv` do mesmo torneio: quantas de cada fauna cada agente coloca, e a relação com o score.

In [ ]:
results_path = ts_path.parent / 'results.csv'
res = pd.read_csv(results_path)
if 'p1_agent' not in res.columns:
    res['p1_agent'] = meta.get('agent_p1', 'P1'); res['p2_agent'] = meta.get('agent_p2', 'P2')

fauna_types = sorted(
    c[len('p1_fauna_'):] for c in res.columns
    if c.startswith('p1_fauna_') and not c.endswith('_total')
)
print('fauna:', fauna_types)

frames = []
for pid in (1, 2):
    sub = pd.DataFrame({'agent': res[f'p{pid}_agent'], 'score': res[f'p{pid}_score']})
    for ft in fauna_types:
        sub[f'fauna_{ft}'] = res[f'p{pid}_fauna_{ft}']
    frames.append(sub)
flong = pd.concat(frames, ignore_index=True)
fcols = [f'fauna_{f}' for f in fauna_types]

mean_by_agent = flong.groupby('agent')[['score'] + fcols].mean().round(2)
mean_by_agent

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
means = flong[fcols].mean().sort_values(ascending=False)
ax.bar([c[len('fauna_'):] for c in means.index], means.values, color='tab:cyan')
ax.set_title('Volume médio de fauna por tipo (por jogador)'); ax.set_ylabel('média por jogo')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

## Pontos marginais por fauna (regressão)

`score ~ intercepto + Σ volumes de fauna`. Estima quantos pontos cada fauna adicional traz, em média.

In [ ]:
X = np.column_stack([np.ones(len(flong))] + [flong[c].values.astype(float) for c in fcols])
y = flong['score'].values.astype(float)
coef, *_ = np.linalg.lstsq(X, y, rcond=None)
reg = pd.Series(coef, index=['(intercepto)'] + fcols).round(2).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
body = reg.drop('(intercepto)')
ax.barh(body.index, body.values, color=['tab:green' if v >= 0 else 'tab:red' for v in body.values])
ax.axvline(0, color='k', linewidth=0.8)
ax.set_title('Pontos marginais por fauna'); ax.set_xlabel('pontos por unidade')
plt.tight_layout(); plt.show()
reg.to_frame('pontos_por_fauna')